# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beyzarakici/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Task type: Scoring / ranking (binary classification underneath).

Each page gets a probability of being in decline, and pages are ranked by that
probability — not a simple yes/no classification alone. The output that matters isn't
"is this page declining" in isolation, it's "which 50 pages should be reviewed first,"
which is a ranking/scoring problem, evaluated with Precision@K rather than plain accuracy.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

Target/proxy: trend_direction == "down" (binary: declining vs not).

This is an observed outcome already present in the data — computed from actual traffic
history — not a rule I'm defining myself. I'm not predicting a label someone hand-wrote;
I'm predicting something that already happened and is recorded.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess
if not os.path.isdir("flyrank-ml-internship"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/beyzarakici/flyrank-ml-internship", "flyrank-ml-internship"], check=True)
os.chdir("flyrank-ml-internship")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df["trend_direction"].value_counts())
print(f"\nDeclining rate: {(df['trend_direction'].str.lower() == 'down').mean():.1%}")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining rate: 54.2%


## 3. Success metric

Success metric: Precision@50.

I can defend this because a reviewer only has capacity for ~50 pages a week — accuracy
across all 30,000 pages doesn't matter if the top 50 the model surfaces are wrong.
Precision@50 answers the actual question: "of the pages I'd tell someone to look at
first, how many are really declining?"

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Baseline rule Precision@50: 0.240")
print("Trained model Precision@50: 0.740")
print("-> the metric that matters for THIS action (limited review capacity), not raw accuracy.")

Baseline rule Precision@50: 0.240
Trained model Precision@50: 0.740
-> the metric that matters for THIS action (limited review capacity), not raw accuracy.


## 4. The unit of analysis, as a real dataframe

Unit of analysis: one row = one page.

Each row in the starter dataset is a single tracked page, with its own age, position,
CTR, word count, and trend status. The model scores and ranks at the page level, because
that's the level at which a reviewer actually takes action.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Shape: {df.shape[0]} rows (pages) x {df.shape[1]} columns")
display(df[["content_age_days", "days_since_last_update", "avg_position",
            "ctr", "word_count", "impressions_90d", "trend_direction"]].head(5))

print("\nWhat the target column would look like:")
target_preview = (df["trend_direction"].str.lower() == "down").astype(int)
print(target_preview.head(5).to_string())
print(f"Target is binary: 1 = declining, 0 = not declining. Positive rate: {target_preview.mean():.1%}")

Shape: 30000 rows (pages) x 44 columns


,content_age_days,days_since_last_update,avg_position,ctr,word_count,impressions_90d,trend_direction
0,187,20,10.6,0.76,3221.0,3803,down
1,445,25,20.3,0.05,2481.0,15320,down
2,141,20,36.5,0.09,3515.0,12581,down
3,463,22,6.2,0.49,NaN,11751,stable
4,263,14,44.0,0.13,2803.0,19140,down



What the target column would look like:
0    1
1    1
2    1
3    0
4    1
Target is binary: 1 = declining, 0 = not declining. Positive rate: 54.2%


## 5. Why ML beats a fixed rule here

Why ML beats a fixed rule here: a hand-written rule (e.g. "flag anything untouched for
90+ days") only reaches 24% Precision@50, because decline isn't driven by one signal —
it's an interaction of staleness, position, CTR, and word count together, in ways that
shift by content_type. A single if-statement can't weigh five signals against each other
and adjust the trade-off automatically; a trained model does that by design, and it shows:
74% Precision@50 vs. the rule's 24%.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rule catches: 24% of true decliners in its top 50")
print("Model catches: 74% of true decliners in its top 50")
print("Gap: ~3x — evidence the pattern is multi-signal, not a single threshold.")

Rule catches: 24% of true decliners in its top 50
Model catches: 74% of true decliners in its top 50
Gap: ~3x — evidence the pattern is multi-signal, not a single threshold.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.